# 05. HCLT(한글 및 한국어 정보처리 학술대회) 논문집의 LLM 에이전트 연구

- 질문: 국내 학회(HCLT 2023~2026)에서 LLM 에이전트 논문은 연도별로 몇 편이고, arXiv(세계 전체·대상 언어)와 키워드군 분포가 어떻게 다른가 (4.6절 후보).
- 입력
  - `preprocessing/output/hclt_papers.csv` — 07: 목차 명부 525편의 제목·저자·쪽수와 선정 판정. 요약·OCR 텍스트는 제외한다.
  - `analysis/output/hclt_tokens.csv` — 논문별 키워드군과 형태소 기반 에이전트 판정. 형태소 텍스트는 제외한다.
  - `analysis/output/arxiv_lemma_families_2023_2026q3.csv` — 09: arXiv LLM 에이전트 논문 전체(2023-01~2026-09)에 같은 사전을 spaCy 표제어로 적용한 키워드군.
  - `verification/review_review_hclt_agent_*.jsonl` — 사람 판정(D23, 검수자 song, 62편 전수).
  - 정본: `verification/hclt_counts_*.json`(07), `verification/hclt_morph_*.json`(09).
- 주의
  - 판정의 정본은 **사람 판정**이다. 형태소 판정은 검수 입력을 고르는 선별기이며 정밀도·재현율만 보고한다.
  - 본 비교 기간은 arXiv와 같은 2023-01~2026-09(D30). HCLT 2026 논문집도 9월까지의 원고다.
  - 키워드군은 "같은 사전의 영문(표제어)·국문(형태소) 판"으로 매칭했다(D26). 영문만 있는 arXiv와 국문 위주인 HCLT의 차이는 사전 번역의 차이를 포함한다.


In [ ]:
import glob, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "verification").exists())   # 노트북은 analysis/notebooks/ 에 있다
DATA = ROOT / "preprocessing/output"; VER = ROOT / "verification"
FIG = ROOT / "analysis/output/figures"; FIG.mkdir(parents=True, exist_ok=True)
TAB = ROOT / "analysis/output/tables"; TAB.mkdir(parents=True, exist_ok=True)
TODAY = pd.Timestamp.today().strftime("%Y%m%d")
try:
    import sys; sys.path.insert(0, str(ROOT / "analysis/scripts")); import figure_style; figure_style.apply_style(); C = figure_style.COLORS; CYC = figure_style.COLOR_CYCLE
except Exception:
    C = {"green": "#46B07B", "orange": "#FF9E02", "teal": "#3A9E8F", "coral": "#E87B4A", "green_dark": "#2D7A53", "orange_dark": "#C47800", "green_light": "#8DD4AA", "orange_light": "#FFD080", "green_pale": "#C8EBDA", "orange_pale": "#FFECC0", "gray": "#7F8C8D", "gray_light": "#BDC3C7", "gray_dark": "#4A4A4A", "text": "#333333"}; CYC = [C["green"], C["orange"], C["teal"], C["coral"]]
    plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
from matplotlib import font_manager as _fm
_ko = [f for f in ("AppleGothic", "NanumGothic", "Malgun Gothic") if f in {x.name for x in _fm.fontManager.ttflist}]
plt.rcParams["font.family"] = ["DejaVu Sans"] + _ko[:1]; plt.rcParams["axes.unicode_minus"] = False   # 그림 글자는 영문이므로 기본 글꼴을 앞에 둔다(한글은 뒤로)
YEARS = [2023, 2024, 2025, 2026]   # D30: 본 비교 기간 2023~2026(HCLT 2026 논문집 포함)

h = pd.read_csv(DATA / "hclt_papers.csv"); OUTA = ROOT / "analysis/output"; SUFFIX = "2023_2026q3"
tok = pd.read_csv(OUTA / "hclt_tokens.csv"); ax = pd.read_csv(OUTA / f"arxiv_lemma_families_{SUFFIX}.csv", dtype={"id": str})
counts = json.loads(Path(sorted(glob.glob(str(VER / "hclt_counts_*.json")))[-1]).read_text())
morph = json.loads(Path(sorted(glob.glob(str(VER / "hclt_morph_*.json")))[-1]).read_text())
rev = pd.read_json(sorted(glob.glob(str(VER / "review_review_hclt_agent_*.jsonl")))[-1], lines=True)
h["human"] = h.id.isin(set(rev[rev.final == "LLM 에이전트"].id)); h = h.merge(tok[["id", "is_llm_agent_morph"] + [c for c in tok.columns if c.startswith("f_")]], on="id")
FAMS = [c[2:] for c in ax.columns if c.startswith("f_")]

# 정본 대조
assert h.groupby("year").size().to_dict() == {int(k): v["papers"] for k, v in counts["human_review"]["by_year"].items()}, "연도별 논문 수가 정본과 다르다"
assert h[h.human].groupby("year").size().to_dict() == {int(k): v for k, v in morph["hclt"]["llm_agent_human_by_year"].items()}, "사람 판정 건수가 정본과 다르다"
assert h[h.is_llm_agent_morph].groupby("year").size().to_dict() == {int(k): v for k, v in morph["hclt"]["llm_agent_morph_by_year"].items()}
assert len(ax) == morph["n"]["arxiv_world"]
assert h.human.sum() == len(rev) == 62 and rev.held.sum() == 0
print("정본 대조 통과: HCLT", len(h), "편 | 사람 판정 LLM 에이전트", int(h.human.sum()), "| arXiv", len(ax))

## 1. 연도별 논문 수와 LLM 에이전트 논문 수

- 계산: 목차 명부의 논문 수, 사람 판정(D23)으로 확정한 LLM 에이전트 수, 비율. 비교용으로 arXiv 세계 전체의 LLM 에이전트 수(정본 `corpus_counts`)를 같은 표에 둔다.
- 입력: `h`(사람 판정 열 human), `verification/corpus_counts_*.json`.
- 주의: HCLT는 자연어처리 학회이고 arXiv는 다섯 분류(cs.AI/CL/CV/CY/LG) 전체라 분모의 성격이 다르다. 비율은 "그 자료 안에서의 비중"이다.


In [ ]:
cc = json.loads(Path(sorted(glob.glob(str(VER / "corpus_counts_*.json")))[-1]).read_text())
t1 = h.groupby("year").agg(papers=("id", "size"), llm_agent=("human", "sum")); t1["share_pct"] = (100 * t1.llm_agent / t1.papers).round(1)
t1["arxiv_llm_agent"] = pd.Series({int(k): v for k, v in cc["llm_agent_by_year"].items()})
t1.loc["합계(2023~2026)"] = t1.loc[YEARS].sum(numeric_only=True); t1.loc["합계(2023~2026)", "share_pct"] = round(100 * t1.loc[YEARS].llm_agent.sum() / t1.loc[YEARS].papers.sum(), 1)
t1.to_csv(TAB / "tab_hclt_year_counts.csv"); print(t1.to_string())

plt.rcParams["font.family"] = ["Nanum Gothic"]
plt.rcParams["axes.unicode_minus"] = False
fig, a1 = plt.subplots(figsize=(6.5, 3.4)); yrs = [2023, 2024, 2025, 2026]; x = np.arange(len(yrs))
a1.bar(x - 0.2, t1.loc[yrs].papers, 0.4, label="전체 논문", color=C["gray_light"], zorder=3); a1.bar(x + 0.2, t1.loc[yrs].llm_agent, 0.4, label="LLM 에이전트 논문(저자 판정)", color=C["green"], zorder=3)
for i, y in enumerate(yrs):
    a1.text(x[i] - 0.2, t1.loc[y].papers + 2, f"{t1.loc[y].papers:.0f}", ha="center", fontsize=7.5, color=C["gray_dark"])
    a1.text(x[i] + 0.2, t1.loc[y].llm_agent + 2, f"{t1.loc[y].llm_agent:.0f} ({t1.loc[y].share_pct}%)", ha="center", fontsize=8, color=C["green_dark"], fontweight="bold")
a1.set_xticks(x); a1.set_xticklabels([str(y) for y in yrs]); a1.set_ylabel("논문 수"); a1.set_title("한글 및 한국어 정보처리 학술대회 논문집\n전체 논문과 LLM 에이전트 논문"); a1.legend(loc="upper left"); a1.set_ylim(0, t1.loc[yrs].papers.max() * 1.15); a1.grid(axis="x", visible=False)
plt.tight_layout(); plt.savefig(FIG / "fig6_hclt_year_counts.png", dpi=200); plt.show()

## 2. 형태소 기반 판정 대 사람 판정

- 계산: 09의 형태소 판정(agent 군 AND llm 군, Kiwi·spaCy 표제어 대조)을 사람 판정과 비교해 정밀도·재현율을 본다. 놓친 논문의 제목을 나열하고, 놓친 이유(agent 군은 있는데 llm 군이 없는가)를 센다.
- 입력: `tok.is_llm_agent_morph`, `h.human`.
- 주의: 검수 입력이 "agent 단어 포함 62편"이므로 그 밖의 논문에서 사람이 LLM 에이전트로 볼 것이 더 있는지는 이 자료로 알 수 없다(한계).


In [ ]:
mv = morph["hclt"]["morph_vs_human"]
tp = int((h.human & h.is_llm_agent_morph).sum()); fp = int((~h.human & h.is_llm_agent_morph).sum()); fn = int((h.human & ~h.is_llm_agent_morph).sum())
assert (tp, fp, fn) == (mv["tp"], mv["fp"], mv["fn"]), "정밀도·재현율 근거 수치가 정본과 다르다"
print(f"정밀도 {tp/(tp+fp):.3f}  재현율 {tp/(tp+fn):.3f}  (TP {tp}, FP {fp}, FN {fn})")
miss = h[h.human & ~h.is_llm_agent_morph][["id", "year", "title_ko", "f_agent", "f_llm"]]; print("형태소 판정이 놓친 논문(사람은 LLM 에이전트):"); print(miss.to_string(index=False))
print(); print("놓친 이유(agent 군, llm 군):"); print(miss[["f_agent", "f_llm"]].value_counts())

## 3. 같은 사전으로 본 키워드군 비율: HCLT 대 arXiv(세계 전체·대상 언어)

- 계산: 17개 키워드군이 LLM 에이전트 논문에 나타나는 비율(%). HCLT는 사람 판정 논문 62편(2023~2026 전체, 2023~2025와 2026을 따로도), arXiv는 세계 전체(2023-01~2026-09, id 접두 기준). 09 정본과 대조한다. 소속 국가 열은 쓰지 않는다(D30).
- 입력: `h`(f_* 열), `ax`(f_* 열).
- 주의: HCLT 2023~2025는 24편이라 한 편이 4.2%p다. 건수 열을 함께 읽는다. agent·llm 군은 정의상 거의 100%라 그림에서 뺀다.


In [ ]:
hall = h[h.human]; hs = h[h.year.isin([2023, 2024, 2025]) & h.human]; h26 = h[(h.year == 2026) & h.human]
rate = lambda d, f: round(100 * d[f"f_{f}"].mean(), 1)
N = {"arXiv world": len(ax), "HCLT 2023-26": len(hall), "HCLT 2023-25": len(hs), "HCLT 2026": len(h26)}; print("n =", N)
t3 = pd.DataFrame({f: {"arXiv world": rate(ax, f), "HCLT 2023-26": rate(hall, f), "HCLT 2023-25": rate(hs, f), "HCLT 2026": rate(h26, f),
                       "n HCLT 23-25": int(hs[f"f_{f}"].sum()), "n HCLT 26": int(h26[f"f_{f}"].sum())} for f in FAMS}).T
for f in FAMS:
    ref = morph["family_rate_pct"][f]
    assert (t3.loc[f, "arXiv world"], t3.loc[f, "HCLT 2023-25"], t3.loc[f, "HCLT 2026"]) == (ref["arxiv_world"], ref["hclt_2023_2025"], ref["hclt_2026"]), f"{f} 비율이 정본과 다르다"
t3 = t3.sort_values("arXiv world", ascending=False); t3.to_csv(TAB / "tab_hclt_vs_arxiv_families.csv"); print(t3.to_string())

show = [f for f in t3.index if f not in ("agent", "llm")]; cols = ["arXiv world", "HCLT 2023-26", "HCLT 2023-25", "HCLT 2026"]; M = t3.loc[show, cols].astype(float)
fig, a = plt.subplots(figsize=(7, 6.2)); im = a.imshow(M.values, cmap="Blues", aspect="auto", vmin=0, vmax=max(40, M.values.max()))
a.set_xticks(range(4)); a.set_xticklabels([f"{c}\n(n={N[c]:,})" for c in cols], fontsize=9); a.set_yticks(range(len(show))); a.set_yticklabels(show)
for i in range(len(show)):
    for j in range(4):
        a.text(j, i, f"{M.values[i, j]:.0f}", ha="center", va="center", fontsize=8, color="white" if M.values[i, j] > 30 else "black")
a.set_title("Keyword-family share (%) among LLM-agent papers\nsame dictionary, lemma/morpheme match"); plt.colorbar(im, ax=a, fraction=0.03, label="%")
plt.tight_layout(); plt.savefig(FIG / "fig7_hclt_vs_arxiv_families.png", dpi=160, bbox_inches="tight"); plt.show()

## 3b. 키워드군 비율: 세계 전체 vs 대상 언어 집합(사람 라벨, D36) vs HCLT

- 계산: 같은 사전·같은 표제어 판정(09)으로 arXiv 세계 전체와 한국어·중국어·일본어 집합(allc의 lang_* = 라벨 '실제 연구 대상')의 키워드군 비율(%)과 편수. HCLT 62편을 나란히 둔다. 원고 표 4(4.4절)의 정본.
- 입력: `ax`(id·f_* 열) + `allc`(lang_* 열), `hall`.
- 주의: 한국어 25편·일본어 18편은 한 편이 4~6%p다. 편수 열을 함께 읽는다. agent·llm 군은 정의상 거의 100%.


In [ ]:
LANGS = ["ko", "zh", "ja"]; LANG_NAME = {"ko": "Korean", "zh": "Chinese", "ja": "Japanese"}
allc = pd.read_csv(DATA / f"corpus_llm_agent_all_{SUFFIX}.csv", dtype={"id": str}, usecols=["id", "lang_ko", "lang_zh", "lang_ja"])
cc = json.loads(Path(sorted(glob.glob(str(VER / "corpus_counts_*.json")))[-1]).read_text())
assert {l: int(allc[f"lang_{l}"].sum()) for l in LANGS} == cc["cjk_lang_by_lang_total"], "대상 언어 집합 크기가 정본과 다르다"
axl = ax.merge(allc[["id"] + [f"lang_{l}" for l in LANGS]], on="id", how="left")
assert len(axl) == len(allc) and axl[[f"lang_{l}" for l in LANGS]].notna().all().all()
sets = {"arXiv world": axl, **{LANG_NAME[l]: axl[axl[f"lang_{l}"]] for l in LANGS}, "HCLT 2023-26": hall}
NS = {k: len(v) for k, v in sets.items()}; print("n =", NS)
t3b = pd.DataFrame({f: {**{k: rate(v, f) for k, v in sets.items()}, **{f"n {k}": int(v[f"f_{f}"].sum()) for k, v in sets.items() if k != "arXiv world"}} for f in FAMS}).T
t3b = t3b.sort_values("arXiv world", ascending=False); t3b.to_csv(TAB / "tab_families_by_set.csv"); print(t3b.to_string())
family_by_set = {"n": NS, "rows": {f: {c: (float(v) if not c.startswith("n ") else int(v)) for c, v in r.items()} for f, r in t3b.iterrows()}}

## 3b-2. 한 장으로 보는 arXiv 대 HCLT — 키워드군 비율 덤벨 그림

- 계산: 표 3b의 세 집합(arXiv 세계 20,107편, arXiv 한국어 집합 25편, HCLT 62편) 비율을 군마다 한 줄에 세 점으로 찍고 HCLT−세계 차이(pp) 순으로 정렬한다. 점 옆에 편수를 적는다. agent·llm 군 제외.
- 읽는 법: 오른쪽(HCLT가 높음)은 질의응답·번역·평가, 왼쪽(HCLT가 낮음)은 강화학습·물리 환경·안전·도구 호출. 한국어 집합(25편)이 HCLT와 같은 쪽에 있는 군이 "국내 연구의 방향".
- 출력: `analysis/output/figures/fig7c_arxiv_vs_hclt_dumbbell.png`


In [ ]:
D = t3b.drop(index=["agent", "llm"]).copy(); D["diff"] = D["HCLT 2023-26"] - D["arXiv world"]; D = D.sort_values("diff")
plt.rcParams["font.family"] = ["Nanum Gothic"]
plt.rcParams["axes.unicode_minus"] = False
fig, ax = plt.subplots(figsize=(8, 6.2)); y = np.arange(len(D))
for i, (f, r) in enumerate(D.iterrows()):
    lo, hi = min(r["arXiv world"], r["HCLT 2023-26"]), max(r["arXiv world"], r["HCLT 2023-26"])
    ax.plot([lo, hi], [i, i], color=C["green_pale"] if r["diff"] >= 0 else C["orange_pale"], lw=5, zorder=1, solid_capstyle="round")
ax.scatter(D["arXiv world"], y, color=C["gray"], s=55, label=f"arXiv 세계 전체 (n={NS['arXiv world']:,})", zorder=3)
ax.scatter(D["Korean"], y, color=C["orange"], s=55, marker="s", label=f"arXiv 한국어 집합 (n={NS['Korean']})", zorder=3)
ax.scatter(D["HCLT 2023-26"], y, color=C["green_dark"], s=75, marker="D", label=f"한글 및 한국어 정보처리 학술대회 논문집\n2023–2026년, 저자 판정 (n={NS['HCLT 2023-26']})", zorder=4)
for i, (f, r) in enumerate(D.iterrows()):
    ax.text(r["HCLT 2023-26"] + (1.3 if r["diff"] >= 0 else -1.3), i, f"{int(r['n HCLT 2023-26'])}", va="center", ha="left" if r["diff"] >= 0 else "right", fontsize=7.5, color=C["green_dark"])
ax.set_yticks(y); ax.set_yticklabels([f.replace("_", " ") for f in D.index]); ax.set_xlabel("키워드군이 나타난 LLM 에이전트 논문의 비율 (%)"); ax.grid(axis="y", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8); ax.set_title("자료별 키워드군 비율: 논문집과 세계 전체의 차이 순")
plt.tight_layout(); plt.savefig(FIG / "fig7c_arxiv_vs_hclt_dumbbell.png", dpi=200, bbox_inches="tight"); plt.show()
print(D[["arXiv world", "Korean", "HCLT 2023-26", "diff", "n HCLT 2023-26"]].to_string())

## 3c. 키워드군 동시출현 lift — 세계 전체 LLM 에이전트 논문 안에서 어떤 군이 함께 나타나는가

- 계산: 군 X와 Y가 같은 논문에 함께 나타나는 비율을 독립일 때의 기대(P(X)·P(Y))로 나눈 lift = P(X∧Y) / (P(X)·P(Y)). 1보다 크면 함께 나타나는 경향, 작으면 서로 피하는 경향. agent·llm 군은 정의상 거의 모든 논문에 있어 뺀다.
- 입력: `axl`(20,107편, f_* 열). 표 3(원고)의 키워드군과 같은 사전·같은 판정.
- 출력: `analysis/output/tables/tab_family_lift.csv`, `analysis/output/figures/fig7b_family_lift.png`
- 주의: lift는 두 군의 동시출현만 보며 토픽이나 인과를 뜻하지 않는다. 비율이 작은 군(transformer 1.5%, diffusion 1.2%)은 lift가 크게 흔들린다.


In [ ]:
fam_show = [f for f in t3b.index if f not in ("agent", "llm")]
M = axl[[f"f_{f}" for f in fam_show]].astype(bool).values; pX = M.mean(axis=0)
Mi = M.astype(int); joint = (Mi.T @ Mi) / len(M)   # 불리언 행렬곱은 논리합이 되므로 정수로 센다
lift = joint / np.outer(pX, pX); np.fill_diagonal(lift, np.nan)
LF = pd.DataFrame(lift, index=fam_show, columns=fam_show).round(2); LF.to_csv(TAB / "tab_family_lift.csv")
pairs = [(fam_show[i], fam_show[j], round(float(lift[i, j]), 2), int((M[:, i] & M[:, j]).sum())) for i in range(len(fam_show)) for j in range(i + 1, len(fam_show))]
pairs = sorted(pairs, key=lambda x: -x[2]); print("함께 나타나는 쌍(상위 10):", pairs[:10]); print("서로 피하는 쌍(하위 10):", pairs[-10:])
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
cmap_brand = LinearSegmentedColormap.from_list("brand_div", [C["orange_dark"], C["orange"], "#FFFFFF", C["green"], C["green_dark"]])
fig, ax = plt.subplots(figsize=(9, 8)); im = ax.imshow(np.clip(lift, 0.25, 4), cmap=cmap_brand, norm=TwoSlopeNorm(vmin=0.25, vcenter=1.0, vmax=4)); ax.grid(False)
ax.set_xticks(range(len(fam_show))); ax.set_xticklabels([f.replace("_", " ") for f in fam_show], rotation=60, ha="right", fontsize=8); ax.set_yticks(range(len(fam_show))); ax.set_yticklabels([f"{f.replace('_', ' ')} ({100*p:.0f}%)" for f, p in zip(fam_show, pX)], fontsize=8)
for i in range(len(fam_show)):
    for j in range(len(fam_show)):
        if i != j: ax.text(j, i, f"{lift[i, j]:.1f}", ha="center", va="center", fontsize=6.5, color="white" if (lift[i, j] > 2.2 or lift[i, j] < 0.5) else C["text"])
cb = plt.colorbar(im, ax=ax, fraction=0.03, ticks=[0.25, 0.5, 1, 2, 4]); cb.set_label("lift (log-like scale; green = co-occur, orange = avoid)"); cb.ax.set_yticklabels(["≤0.25", "0.5", "1", "2", "≥4"])
ax.set_title(f"Keyword-family co-occurrence lift within LLM-agent papers (n={len(M):,}, 2023-01 to 2026-09)")
plt.tight_layout(); plt.savefig(FIG / "fig7b_family_lift.png", dpi=200, bbox_inches="tight"); plt.show()
family_lift = {"families": fam_show, "share_pct": {f: round(100 * float(p), 1) for f, p in zip(fam_show, pX)}, "lift": {a: {b: (None if np.isnan(LF.loc[a, b]) else float(LF.loc[a, b])) for b in fam_show} for a in fam_show}, "top_pairs": pairs[:15], "bottom_pairs": pairs[-15:]}

## 4. 집계 저장

- 계산: 논문별 판정으로 구한 편수·키워드군 비율을 저장한다.
- 입력: 공개한 판정 열과 arXiv 키워드군 자료.
- 주의: 학술대회 요약·형태소 텍스트는 배포하지 않으며, 추출 오류를 교정한 재분석 결과는 아니다.


In [ ]:
json.dump({"generated": TODAY, "notebook": "analysis/notebooks/05_hclt.ipynb", "n": {"hclt_2023_2025": int(len(hs)), "hclt_2026": int(len(h26))}, "family_by_set": family_by_set, "family_lift": family_lift,
           "year_counts": {str(k): {c: (float(v) if c == "share_pct" else int(v)) for c, v in row.items() if pd.notna(v)} for k, row in t1.iterrows()}},
          open(VER / f"hclt_notebook_{TODAY}.json", "w"), ensure_ascii=False, indent=1); print("saved", VER / f"hclt_notebook_{TODAY}.json")